In [5]:
empty code block

SyntaxError: invalid syntax (ipython-input-521930699.py, line 1)

In [7]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


In [8]:
!ls /content/drive

MyDrive


In [10]:

from pathlib import Path

# Search your whole MyDrive for the file
hits = list(Path("/content/drive/MyDrive").rglob("sr2025.zip"))
hits


[PosixPath('/content/drive/MyDrive/visualization2/data/sr2025.zip')]

In [11]:
# Toronto 311 Final Project – Python Visual 1
# Exclude Status == 'Canceled' to reflect actionable public demand.
# Use the Original Service Request Type (customer phrasing), per dataset docs.

import io, os, re, zipfile
from datetime import datetime
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

DATA_ZIP = "/content/drive/MyDrive/visualization2/data/sr2025.zip"
OUT_DIR  = "outputs"
os.makedirs(OUT_DIR, exist_ok=True)

def load_311_from_zip(zip_path: str) -> pd.DataFrame:
    """
    Load the single CSV contained in the given ZIP file.

    Behavior:
      - Verifies there's exactly one CSV in the ZIP.
      - Tries the fast C engine first.
      - Falls back to the Python engine (more tolerant) without low_memory.
    """
    import zipfile, io
    from pathlib import Path

    p = Path(zip_path)
    if not p.exists():
        raise FileNotFoundError(
            f"Could not find {p}. Working directory: {Path.cwd()} "
            "→ Ensure the ZIP is in ./data (or update DATA_ZIP)."
        )

    with zipfile.ZipFile(p, "r") as zf:
        csv_members = [n for n in zf.namelist() if n.lower().endswith(".csv")]
        if len(csv_members) == 0:
            raise ValueError("No CSV file found in the ZIP.")
        if len(csv_members) > 1:
            raise ValueError(
                f"Expected exactly 1 CSV in the ZIP, found {len(csv_members)}: {csv_members}"
            )

        name = csv_members[0]

        # 1) Try fast C engine
        try:
            with zf.open(name) as f:
                df = pd.read_csv(
                    f,
                    encoding="utf-8-sig",  # handles potential BOM
                    low_memory=False       # supported by C engine
                )
            print(f"Loaded {name}: {len(df):,} rows (C engine)")
            return df

        except Exception as e:
            print(f"{name}: C engine failed ({e}). Retrying with Python engine...")

            # 2) Fallback: Python engine (tolerant parsing; do NOT pass low_memory)
            with zf.open(name) as f:
                text = io.TextIOWrapper(f, encoding="utf-8", errors="replace")
                df = pd.read_csv(
                    text,
                    engine="python",
                    sep=None,            # auto-detect delimiter
                    quotechar='"',
                    escapechar='\\',     # handle stray quotes/commas
                    on_bad_lines="warn"  # 'skip' to silence; 'error' to be strict
                )
            print(f"Loaded {name}: {len(df):,} rows (Python engine)")
            return df

def clean_and_filter(df: pd.DataFrame) -> pd.DataFrame:
    """
    Parse dates, keep 2025 rows, exclude Canceled, and normalize column names across schema variants.

    Supports both:
      - 'Service Request Creation Date and Time' / 'Service Request Status' / 'Original Service Request Type'
      - 'Creation Date' / 'Status' / 'Service Request Type'
    """
    # 1) Normalize whitespace around column names
    df = df.copy()
    df.columns = [c.strip() for c in df.columns]

    # 2) Identify timestamp column
    ts_candidates = [
        "Service Request Creation Date and Time",  # older/alternate schema
        "Creation Date"                            # your schema
    ]
    ts_col = next((c for c in ts_candidates if c in df.columns), None)
    if ts_col is None:
        raise KeyError(
            "Could not find a timestamp column. Looked for: "
            f"{ts_candidates}. Available: {list(df.columns)[:30]}"
        )

    # 3) Identify status column
    status_candidates = [
        "Service Request Status",  # older/alternate schema
        "Status"                   # your schema
    ]
    status_col = next((c for c in status_candidates if c in df.columns), None)

    # 4) Identify request type column
    type_candidates = [
        "Original Service Request Type",  # older/alternate schema (customer phrasing)
        "Service Request Type"            # your schema
    ]
    type_col = next((c for c in type_candidates if c in df.columns), None)
    if type_col is None:
        raise KeyError(
            "Could not find a request type column. Looked for: "
            f"{type_candidates}. Available: {list(df.columns)[:30]}"
        )

    # 5) Parse timestamp (date or datetime)
    #    'Creation Date' may be date-only; coerce errors -> NaT is fine
    df["created_dt"] = pd.to_datetime(df[ts_col], errors="coerce")
    # Filter to year 2025
    df = df[df["created_dt"].dt.year == 2025].copy()

    # 6) Exclude 'Canceled' if we have a status column
    if status_col and status_col in df.columns:
        # guard against NaNs and various casing
        df = df[df[status_col].astype(str).str.lower() != "canceled"]

    # 7) Normalize the request type labels
    df[type_col] = (
        df[type_col]
          .astype(str)
          .str.strip()
          .str.replace(r"\s+", " ", regex=True)
    )

    # 8) Return with a consistent name for the request type
    return df.rename(columns={type_col: "request_type"})

def make_viz1(df: pd.DataFrame, out_path: str, top_n: int = 8):
    """Line chart of monthly volume for Top-N request types."""
    df["month"] = df["created_dt"].dt.to_period("M").dt.to_timestamp()

    # Top-N overall in 2025
    top_types = (
        df.groupby("request_type", dropna=False)
          .size().sort_values(ascending=False).head(top_n).index.tolist()
    )
    dft = df[df["request_type"].isin(top_types)]
    monthly = (
        dft.groupby(["month", "request_type"])
           .size().reset_index(name="requests")
    )

    sns.set_theme(style="whitegrid")
    plt.figure(figsize=(11, 6))
    ax = sns.lineplot(
        data=monthly, x="month", y="requests", hue="request_type",
        style="request_type", palette="colorblind", linewidth=2.0, marker="o"
    )
    ax.set_title("Toronto 311 – Top Service Request Types by Month (2025)", fontsize=14, weight="bold")
    ax.set_xlabel("Month (2025)")
    ax.set_ylabel("Requests (count)")
    ax.tick_params(axis='x', rotation=0)
    ax.legend(title="Request type", bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)
    plt.tight_layout()
    plt.savefig(out_path, dpi=300)
    plt.close()

def export_ward_counts(df: pd.DataFrame, out_csv: str):
    """Export 2025 counts by ward number for Tableau. Tableau will do population normalization."""
    ward_col = "Service Request Ward"
    if ward_col not in df.columns:
        # look for 'Ward'
        candidates = [c for c in df.columns if "Ward" in c]
        if not candidates:
            raise KeyError(
                "Could not find a 'Ward' column for export. "
                f"Available columns (first 30): {list(df.columns)[:30]}"
            )
        ward_col = candidates[0]

    # Extract ward number (handles “Ward 10 – Spadina–Fort York”, “Spadina–Fort York (10)”, etc.)
    def extract_ward_num(s):
        if pd.isna(s):
            return None
        m = re.search(r"(\d{1,2})", str(s))
        return int(m.group(1)) if m else None

    df["ward_num"] = df[ward_col].apply(extract_ward_num)
    ward_counts = (
        df.dropna(subset=["ward_num"])
          .groupby("ward_num")
          .size().reset_index(name="requests_2025")
          .sort_values("ward_num")
    )
    ward_counts.to_csv(out_csv, index=False)

def alt_text_for_viz1():
    """Alt text string for accessibility."""
    return (
        "Line chart showing monthly counts in 2025 for the top Toronto 311 request types. "
        "Each top category forms a separate line with markers. Several categories peak in winter and spring; "
        "others rise in summer. The legend lists request types; y-axis is request count; x-axis shows months Jan–Dec 2025."
    )

if __name__ == "__main__":
    df_raw = load_311_from_zip(DATA_ZIP)
    # Quick peek to confirm key columns are present
    print("Columns (first 30):", list(df_raw.columns)[:30])
    print("Rows:", len(df_raw))

    df = clean_and_filter(df_raw)
    print("Rows after 2025 & status filter:", len(df))

    make_viz1(df, os.path.join(OUT_DIR, "viz1_top_types_2025.png"), top_n=8)
    export_ward_counts(df, os.path.join(OUT_DIR, "311_2025_by_ward_counts.csv"))

    print("Saved:", os.path.join(OUT_DIR, "viz1_top_types_2025.png"))
    print("Saved:", os.path.join(OUT_DIR, "311_2025_by_ward_counts.csv"))
    print(alt_text_for_viz1())

SR2025.csv: C engine failed (Error tokenizing data. C error: Expected 9 fields in line 45, saw 10
). Retrying with Python engine...


/tmp/ipython-input-158011702.py:62: ParserWarning: Skipping line 45: Expected 9 fields in line 45, saw 10

  df = pd.read_csv(
/tmp/ipython-input-158011702.py:62: ParserWarning: Skipping line 94: Expected 9 fields in line 94, saw 10

  df = pd.read_csv(
/tmp/ipython-input-158011702.py:62: ParserWarning: Skipping line 104: Expected 9 fields in line 104, saw 10

  df = pd.read_csv(
/tmp/ipython-input-158011702.py:62: ParserWarning: Skipping line 149: Expected 9 fields in line 149, saw 10

  df = pd.read_csv(
/tmp/ipython-input-158011702.py:62: ParserWarning: Skipping line 212: Expected 9 fields in line 212, saw 10

  df = pd.read_csv(
/tmp/ipython-input-158011702.py:62: ParserWarning: Skipping line 241: Expected 9 fields in line 241, saw 10

  df = pd.read_csv(
/tmp/ipython-input-158011702.py:62: ParserWarning: Skipping line 263: Expected 9 fields in line 263, saw 10

  df = pd.read_csv(
/tmp/ipython-input-158011702.py:62: ParserWarning: Skipping line 278: Expected 9 fields in line 278, 

Loaded SR2025.csv: 464,080 rows (Python engine)
Columns (first 30): ['Creation Date', 'Status', 'First 3 Chars of Postal Code', 'Intersection Street 1', 'Intersection Street 2', 'Ward', 'Service Request Type', 'Division', 'Section']
Rows: 464080
Rows after 2025 & status filter: 464080
Saved: outputs/viz1_top_types_2025.png
Saved: outputs/311_2025_by_ward_counts.csv
Line chart showing monthly counts in 2025 for the top Toronto 311 request types. Each top category forms a separate line with markers. Several categories peak in winter and spring; others rise in summer. The legend lists request types; y-axis is request count; x-axis shows months Jan–Dec 2025.
